In [11]:
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim
import ssl
import time
import base64

def connect_to_vcenter(host, user, pwd):
    context = ssl._create_unverified_context()
    si = SmartConnect(host=host, user=user, pwd=pwd, sslContext=context)
    return si

def get_vm_by_name(si, vm_name):
    content = si.RetrieveContent()
    obj_view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
    vm_list = obj_view.view
    obj_view.Destroy()
    for vm in vm_list:
        if vm.name == vm_name:
            return vm
    return None

def get_vm_status(vm):
    summary = vm.summary
    config = summary.config
    print(f"Vcenter Info VM Name    : {vm.name}")
    print(f"Vcenter Info CPU Count  : {config.numCpu}")
    print(f"Vcenter Info Memory Size: {config.memorySizeMB} MB")

def wait_for_task(task):
    while task.info.state not in [vim.TaskInfo.State.success, vim.TaskInfo.State.error]:
        time.sleep(1)
    if task.info.state == vim.TaskInfo.State.error:
        print(f"Task failed: {task.info.error}")
        raise task.info.error
    return task.info.result

def power_off_vm(vm):
    if vm.runtime.powerState == vim.VirtualMachinePowerState.poweredOn:
        task = vm.PowerOffVM_Task()
        wait_for_task(task)

def power_on_vm(vm):
    if vm.runtime.powerState == vim.VirtualMachinePowerState.poweredOff:
        task = vm.PowerOnVM_Task()
        wait_for_task(task)

def reconfigure_vm(vm, num_cpu=None, memory_size_mb=None):
    if num_cpu is None and memory_size_mb is None:
        print("No changes specified. Please provide CPU and/or Memory values to update.")
        return

    spec = vim.vm.ConfigSpec()
    if num_cpu is not None:
        spec.numCPUs = num_cpu
    if memory_size_mb is not None:
        spec.memoryMB = memory_size_mb*1024

    task = vm.ReconfigVM_Task(spec=spec)
    wait_for_task(task)

def main():
    vcenter_host = "dcmidvmvcs025.nam.corp.gm.com"
    vcenter_user = base64.b64decode('YXNfanp3eGZnQG5hbS5jb3JwLmdtLmNvbQ==').decode()
    vcenter_password = base64.b64decode('ZTh5ckxhbk5ZcUo2eFNaZA==').decode()
    vm_name = "dcwidavpgs1297"
    new_cpu_count = 16  # Yeni CPU sayısı
    new_memory_size_mb = 32  # Yeni bellek boyutu (MB)

    si = connect_to_vcenter(vcenter_host, vcenter_user, vcenter_password)
    
    try:
        vm = get_vm_by_name(si, vm_name)
        if vm is None:
            print(f"VM {vm_name} not found.")
            return

        # Mevcut durumu göster
        get_vm_status(vm)

        # VM'i kapat
        power_off_vm(vm)

        # VM'i yeniden yapılandır
        reconfigure_vm(vm, new_cpu_count, new_memory_size_mb)

        # VM'i aç
        power_on_vm(vm)

        # Yeni durumu göster
        get_vm_status(vm)

    finally:
        Disconnect(si)

if __name__ == "__main__":
    main()


VM dcwidavpgs1297 not found.


** KOD GELISTIRME:   

Bir sonraki adim olarak bir csv dosyasindan sunucu adi,vcenter,cpu,memry bilgilerini okuyarak otomatik olarak yapacak

In [13]:
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim
import ssl
import time
import base64

def connect_to_vcenter(host, user, pwd):
    context = ssl._create_unverified_context()
    si = SmartConnect(host=host, user=user, pwd=pwd, sslContext=context)
    return si

def get_vm_by_name(si, vm_name):
    content = si.RetrieveContent()
    obj_view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
    vm_list = obj_view.view
    obj_view.Destroy()
    for vm in vm_list:
        if vm.name == vm_name:
            return vm
    return None

def get_vm_status(vm):
    summary = vm.summary
    config = summary.config
    print(f"Vcenter Info VM Name    : {vm.name}")
    print(f"Vcenter Info CPU Count  : {config.numCpu}")
    print(f"Vcenter Info Memory Size: {config.memorySizeMB} MB")

def wait_for_task(task):
    while task.info.state not in [vim.TaskInfo.State.success, vim.TaskInfo.State.error]:
        time.sleep(1)
    if task.info.state == vim.TaskInfo.State.error:
        print(f"Task failed: {task.info.error}")
        raise task.info.error
    return task.info.result

def power_off_vm(vm):
    if vm.runtime.powerState == vim.VirtualMachinePowerState.poweredOn:
        task = vm.PowerOffVM_Task()
        wait_for_task(task)

def power_on_vm(vm):
    if vm.runtime.powerState == vim.VirtualMachinePowerState.poweredOff:
        task = vm.PowerOnVM_Task()
        wait_for_task(task)

def reconfigure_vm(vm, num_cpu=None, memory_size_mb=None):
    if num_cpu is None and memory_size_mb is None:
        print("No changes specified. Please provide CPU and/or Memory values to update.")
        return

    spec = vim.vm.ConfigSpec()
    if num_cpu is not None:
        spec.numCPUs = num_cpu
    if memory_size_mb is not None:
        spec.memoryMB = memory_size_mb*1024

    task = vm.ReconfigVM_Task(spec=spec)
    wait_for_task(task)

def process_vm(vm_name, si, new_cpu_count, new_memory_size_mb):
    vm = get_vm_by_name(si, vm_name)
    if vm is None:
        print(f"VM {vm_name} not found.")
        return

    # Mevcut durumu göster
    get_vm_status(vm)

    # VM'i kapat
    power_off_vm(vm)

    # VM'i yeniden yapılandır
    reconfigure_vm(vm, new_cpu_count, new_memory_size_mb)

    # VM'i aç
    power_on_vm(vm)

    # Yeni durumu göster
    get_vm_status(vm)

def main():
    vcenter_host = "dcmidvmvcs025.nam.corp.gm.com"
    vcenter_user = base64.b64decode('YXNfanp3eGZnQG5hbS5jb3JwLmdtLmNvbQ==').decode()
    vcenter_password = base64.b64decode('dUJTWFp6NXdGUFlIZ3Bmaw=').decode()
    new_cpu_count = 16  # Yeni CPU sayısı
    new_memory_size_mb = 32  # Yeni bellek boyutu (MB)
    vm_names = [
 ""
    ]  # İşlem yapılacak VM isimleri listesi


    si = connect_to_vcenter(vcenter_host, vcenter_user, vcenter_password)
    
    try:
        for vm_name in vm_names:
            process_vm(vm_name, si, new_cpu_count, new_memory_size_mb)
            print("="*50)
    
    finally:
        Disconnect(si)

if __name__ == "__main__":
    main()


Vcenter Info VM Name    : dcmidavpgs1361
Vcenter Info CPU Count  : 8
Vcenter Info Memory Size: 16384 MB
Vcenter Info VM Name    : dcmidavpgs1361
Vcenter Info CPU Count  : 16
Vcenter Info Memory Size: 32768 MB
Vcenter Info VM Name    : dcmidavpgs1362
Vcenter Info CPU Count  : 8
Vcenter Info Memory Size: 16384 MB
Vcenter Info VM Name    : dcmidavpgs1362
Vcenter Info CPU Count  : 16
Vcenter Info Memory Size: 32768 MB
Vcenter Info VM Name    : dcmidavpgs1381
Vcenter Info CPU Count  : 8
Vcenter Info Memory Size: 16384 MB
Vcenter Info VM Name    : dcmidavpgs1381
Vcenter Info CPU Count  : 16
Vcenter Info Memory Size: 32768 MB
Vcenter Info VM Name    : dcmidavpgs1382
Vcenter Info CPU Count  : 8
Vcenter Info Memory Size: 16384 MB
Vcenter Info VM Name    : dcmidavpgs1382
Vcenter Info CPU Count  : 16
Vcenter Info Memory Size: 32768 MB
Vcenter Info VM Name    : dcmidavpgs1383
Vcenter Info CPU Count  : 8
Vcenter Info Memory Size: 16384 MB
Vcenter Info VM Name    : dcmidavpgs1383
Vcenter Info CPU Co

In [15]:
sh /home/jzwxfg/Documents/scripts/SR35051510/sr.sh > /home/jzwxfg/Documents/scripts/SR35051510/AfterAddCPUmem.out

SyntaxError: invalid syntax (2838346675.py, line 1)